# 05 · So sánh kết quả DIRE giữa các run

Notebook đọc JSON đã lưu để lập bảng train/eval theo generator và run ID. Mặc định không gọi Modal, không đọc `.npy`, không chạy GPU và không sửa kết quả.

Mỗi run cần `train.json`, `eval.json`, và tùy chọn `report.json` trong `runs/rebuild/<run_id>/`. Đặt `DIRE_RUNS_ROOT=/zero/runs/rebuild` nếu kernel có mount Volume; khi chạy local, tải các JSON cần so sánh về trước.


## 1. Chọn các run

Sửa `RUN_IDS`, hoặc đặt `DIRE_RUN_IDS=run-a,run-b`. `DIRE_RUNS_ROOT` là thư mục **chứa** các thư mục theo run ID.


In [ ]:
import json
import os
from pathlib import Path
from html import escape

RUNS_ROOT = Path(os.environ.get("DIRE_RUNS_ROOT", "runs/rebuild"))
RUN_IDS = [x.strip() for x in os.environ.get(
    "DIRE_RUN_IDS", "rebuild-2026-10-14"
).split(",") if x.strip()]
print("Thư mục kết quả:", RUNS_ROOT.resolve())
print("Run IDs:", RUN_IDS)


## 2. Đọc ledger an toàn

Thiếu file được báo rõ, không tự điền metric. `by_generator` trong `train.json` là kết quả detector; `by_tag` trong `eval.json` là metric ở split được chấm.


In [ ]:
def load_json(path):
    path = Path(path)
    if not path.is_file():
        print("THIẾU:", path)
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as exc:
        print(f"LỖI ĐỌC {path}: {exc}")
        return {}

runs = {}
for run_id in RUN_IDS:
    root = RUNS_ROOT / run_id
    runs[run_id] = {
        "root": root,
        "train": load_json(root / "train.json"),
        "eval": load_json(root / "eval.json"),
        "report": load_json(root / "report.json"),
    }


## 3. Bảng training

Hiển thị ACC/AP/AUC, cỡ mẫu và epoch tốt nhất theo generator. Giá trị thiếu giữ nguyên là `None`.


In [ ]:
train_rows = []
for run_id, bundle in runs.items():
    for tag, row in (bundle["train"].get("by_generator") or {}).items():
        train_rows.append({
            "run_id": run_id, "detector": tag,
            "ACC": row.get("acc"), "ACC@0.5": row.get("acc_at_0.5"),
            "AP": row.get("ap"), "AUC": row.get("auc"),
            "n": row.get("n"), "best_epoch": row.get("best_epoch"),
        })

def show_table(rows, columns):
    if not rows:
        print("Chưa có dòng kết quả trong các run đã chọn.")
        return
    try:
        from IPython.display import HTML, display
        head = "".join(f"<th>{escape(str(c))}</th>" for c in columns)
        body = "".join(
            "<tr>" + "".join(f"<td>{escape(str(row.get(c, '')))}</td>" for c in columns) + "</tr>"
            for row in rows
        )
        display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))
    except ImportError:
        print(" | ".join(columns))
        for row in rows:
            print(" | ".join(str(row.get(c, "")) for c in columns))

show_table(train_rows, ["run_id", "detector", "ACC", "ACC@0.5", "AP", "AUC", "n", "best_epoch"])


## 4. Bảng evaluation

So sánh cùng generator, split và cỡ mẫu; `gate` phải là `passed` trước khi trích kết quả chính thức.


In [ ]:
eval_rows = []
for run_id, bundle in runs.items():
    for tag, row in (bundle["eval"].get("by_tag") or {}).items():
        eval_rows.append({
            "run_id": run_id, "detector": tag, "split": row.get("split"),
            "ACC": row.get("acc"), "AP": row.get("ap"), "AUC": row.get("auc"),
            "n": row.get("n"), "gate": row.get("gate"),
        })
show_table(eval_rows, ["run_id", "detector", "split", "ACC", "AP", "AUC", "n", "gate"])


## 5. Metadata và khả năng so sánh

Khác `maps_root` có thể là khác dataset/map. Chỉ so trực tiếp khi xác nhận cùng dữ liệu, protocol, seed và split.


In [ ]:
meta_rows = []
for run_id, bundle in runs.items():
    report = bundle["report"]
    warnings = report.get("warnings") or []
    meta_rows.append({
        "run_id": run_id,
        "maps_root": report.get("maps_root", "(report thiếu)"),
        "label": report.get("label", "(report thiếu)"),
        "n_warnings": len(warnings),
        "warnings": "; ".join(str(x) for x in warnings[:3]),
    })
show_table(meta_rows, ["run_id", "maps_root", "label", "n_warnings", "warnings"])


## 6. Cách diễn giải

- Đọc ACC cùng AP/AUC và số mẫu; không so ACC giữa split/cỡ mẫu khác nhau mà không ghi chú.
- Midjourney hiện có 100 ảnh test so với 1.000 ảnh thật trong snapshot — tỷ lệ 10:1.
- Detector StyleGAN không chứng minh generalization sang DDPM/Midjourney; evaluator hiện tại chưa tạo ma trận cross-generator đầy đủ.
- `train --epochs 40` là pilot scratch, không phải protocol paper đã xác nhận.
- Các bảng phản ánh JSON đã lưu; notebook không tính lại metric từ score thô.


## Tải JSON từ Modal Volume về local

Thay ID/thư mục đích nếu cần. Lệnh chỉ tải artifact, không sửa Volume.

```bash
modal volume get dire-zero runs/rebuild/rebuild-2026-10-14 ./runs/rebuild/rebuild-2026-10-14
```

Sau đó đặt `DIRE_RUNS_ROOT=./runs/rebuild` hoặc sửa `RUNS_ROOT` ở ô đầu.
